In [9]:
# Requires train_features to run

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import sys; print(sys.executable)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split 
from sklearn.metrics import log_loss, roc_auc_score, log_loss
from sklearn.metrics import accuracy_score
from sklearn import preprocessing
from sklearn.metrics import brier_score_loss
df_master = pd.read_parquet("../data/train_features.parquet")

/Users/faithakanbi/Documents/American-Express-1A-second-look-lending/.venv/bin/python


In [11]:
train_mask = df_master["WEEK_NUM"] < 74
test_mask = df_master["WEEK_NUM"] >= 74

# Using all the same features as the rest of the models
feature_list = [
    c for c in df_master.columns
    if c not in ["case_id", "WEEK_NUM", "target"]
]

In [12]:
X = df_master[feature_list]
y = df_master["target"]

# Training and Testing Split
X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print(X_train.shape)
print(X_test.shape)

(804297, 92)
(195703, 92)


In [13]:
# Filling the NaN values for Logistic Regression
train_medians = X_train.median()
X_train_filled = X_train.fillna(train_medians)
X_test_filled = X_test.fillna(train_medians)
thin_file = df_master.loc[y_test.index, "thin_file"]

#scaling values to reduce num of iterations in the LR
scaler = preprocessing.StandardScaler().fit(X_train_filled)
X_train_scaled = scaler.transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

In [14]:
def train_LR(X_train, y_train, X_test, y_test, c=0.1):
    model = LogisticRegression(C=c, max_iter=1500)
    model.fit(X_train, y_train)
    prob_predictions = model.predict_proba(X_test)
    l_loss = log_loss(y_test, prob_predictions)
    class_label_predictions = model.predict(X_test)

    val_probs = model.predict_proba(X_test_scaled)[:, 1]

    acc_score = accuracy_score(y_test, class_label_predictions)
    overall_auc = roc_auc_score(y_test, prob_predictions[:, 1])
    established_auc = roc_auc_score(y_test[thin_file == 0],prob_predictions[thin_file == 0, 1])
    thin_file_auc = roc_auc_score(y_test[thin_file == 1],prob_predictions[thin_file == 1, 1])
    brier = brier_score_loss(y_test, prob_predictions[:, 1])

    return l_loss, acc_score, overall_auc, established_auc, thin_file_auc, brier, val_probs

In [15]:
best_c = 0.01

final_results = train_LR(
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
    c=best_c
)


In [16]:
# Gets the applicants that actually repaid their loans
y = y_test
repaid = y == 0

val_df = df_master.loc[X_test.index]
mid_week = val_df["WEEK_NUM"].median()
early = (val_df["WEEK_NUM"] <= mid_week).values
late = ~early

# approving applicants who predicted default prob is below 20%
approve = final_results[6] < 0.20

# getting the totals for who defaulted and repaid
approved_repaid = (approve & repaid & late).sum()
approved_defaulted = (approve & ~repaid & late).sum()

# using the calculation from Justin's PR
profit = 2000 * approved_repaid - 8000 * approved_defaulted

In [17]:
predictive_summary = pd.DataFrame({
    "Metric": [
        "Overall AUC",
        "Established AUC",
        "Thin-file AUC",
        "Log Loss",
        "Accuracy",
        "Brier Score",
        "Approved Repayers",
        "Approved Defaulters",
        "Profit ($)"
    ],
    "Logistic Regression (Baseline)": [
        final_results[2],
        final_results[3],
        final_results[4],
        final_results[0],
        final_results[1],
        final_results[5],
        approved_repaid,
        approved_defaulted,
        profit
    ]
})

predictive_summary = predictive_summary.style.format({
        "Logistic Regression (Baseline)": "{:,.4f}"
    })
display(predictive_summary)

,Metric,Logistic Regression (Baseline)
0,Overall AUC,0.7609
1,Established AUC,0.7867
2,Thin-file AUC,0.7118
3,Log Loss,0.3995
4,Accuracy,0.8323
5,Brier Score,0.1249
6,Approved Repayers,"47,980.0000"
7,Approved Defaulters,"4,277.0000"
8,Profit ($),"61,744,000.0000"
